In [1]:
import cv2
import numpy as np
import os

DCP

In [2]:
def dark_channel(img,size=22):
    b,g,r=cv2.split(img)
    min=cv2.min(cv2.min(r,g),b)
    kernel=cv2.getStructuringElement(cv2.MORPH_RECT,(size,size))
    return cv2.erode(min,kernel)

def atm_light(img,dark):
    flat_img=img.reshape(-1,3)
    flat_dark=dark.reshape(-1)

    indices=np.argsort(flat_dark)[-1000:]
    A=np.mean(flat_img[indices],axis=0)
    return A

def tm(img,A,omega=0.75,size=22):
    norm=img/A
    dark=dark_channel(norm)
    omega=np.mean(dark)
    return 1-omega*dark_channel(norm)
    
def recover(img,t,A):
    t_min=np.percentile(t,5)
    t=np.clip(t,t_min,1)
    J=np.zeros_like(img,dtype=np.float32)

    for i in range(3):
        J[:,:,i]=((img[:,:,i]-A[i])/t) + A[i]

    return np.clip(J,0,255).astype(np.uint8)

CLAHE

In [3]:
def clahe(img):
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    clahe=cv2.createCLAHE(clipLimit=2.5,tileGridSize=(22,16))
    l=clahe.apply(l)
    lab=cv2.merge((l,a,b))
    return cv2.cvtColor(lab,cv2.COLOR_LAB2BGR)

Color Correction

In [4]:
def color_balance(img):
    result=img.astype(np.float32)
    result[:,:,2]*=1.2
    result[:,:,0]*=0.85

    return np.clip(result,0,255).astype(np.uint8)

Main function

In [5]:
def main(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for file in os.listdir(input_folder):
        if not file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            continue

        input_path = os.path.join(input_folder, file)

        img = cv2.imread(input_path)
        if img is None:
            print(f"Skipping: {file}")
            continue

        img = img.astype(np.float32)

        img_cb = color_balance(img)
        dark = dark_channel(img_cb)
        A = atm_light(img_cb, dark)
        t = tm(img_cb, A)
        img_dcp = recover(img_cb, t, A)
        img_clahe = clahe(img_dcp)

        save_path = os.path.join(output_folder, file)
        cv2.imwrite(save_path, img_clahe)

        print(f"Saved: {save_path}")

In [7]:
main(r"C:\Users\aaish\Downloads\calib_data_front_5\calib_data_front\right",r"C:\Users\aaish\OneDrive\Desktop\pipeline_2\right")

Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_000.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_001.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_002.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_003.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_004.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_005.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_006.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_007.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_008.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_009.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_010.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_011.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_012.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_2\right\img_013.png
Saved: C:\Users\aaish\OneDrive\Desktop\pipeline_

In [76]:
def main(input,output):
    img=cv2.imread(input).astype(np.float32)
    
    img_cb=color_balance(img)
    
    dark=dark_channel(img_cb)
    A=atm_light(img_cb,dark)
    t=tm(img_cb,A)
    img_dcp=recover(img_cb,t,A)

    img_clahe=clahe(img_dcp)

    os.makedirs(output, exist_ok=True)
    filename = os.path.basename(input)
    name, ext = os.path.splitext(filename)
    save_path = os.path.join(output, name + "_dynamic" + ext)
    cv2.imwrite(save_path, img_clahe)
    cv2.imshow('Original',cv2.imread(input))
    cv2.imshow('Transformed',img_clahe)
    cv2.waitKey(0)
    cv2.destroyAllWindows()